# Init

In [0]:
import pyspark.sql.functions as F 
from pyspark.sql.window import Window

#The Transformation Logic

In [0]:
query = """
-- Deduplicate dimension tables first to prevent cartesian products
WITH dedupe_products AS (
  SELECT 
    product_number,
    product_key,
    ROW_NUMBER() OVER (PARTITION BY product_number ORDER BY product_key DESC) as rn
  FROM workspace_1.gold.dim_products
),
dedupe_customers AS (
  SELECT 
    customer_id,
    customer_key,
    ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY customer_key DESC) as rn
  FROM workspace_1.gold.dim_customers
)
SELECT
    sd.order_number,
    pr.product_key,
    cu.customer_key,
    sd.order_date,
    sd.ship_date,
    sd.due_date,
    sd.sales_amount,
    sd.quantity,
    sd.price
FROM workspace_1.silver.crm_sales sd
LEFT JOIN dedupe_products pr
    ON sd.product_number = pr.product_number AND pr.rn = 1
LEFT JOIN dedupe_customers cu
    ON sd.customer_id = cu.customer_id AND cu.rn = 1
WHERE sd.sales_amount IS NULL OR sd.sales_amount >= 0;  -- Filter out only negative sales, keep nulls
"""
df = spark.sql(query)

In [0]:
df.limit(10).display()

# Writing Gold Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace_1.gold.fact_sales")